# VascularAge — Phase 1 Rebuild: Computational Engine and Source Qualification

This notebook performs qualification only. It does **not** calculate real-PWDB cross-age distances, aliases, alias prevalence, age-identifiability surfaces, compensation vectors, or measurement-rescue results.

The rebuild uses VascuQuest as the authority for waveform missing/padding semantics. Run **Runtime → Run all** without editing cells.


In [9]:
from __future__ import annotations
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

ALLOW_BIOLOGICAL_ENDPOINTS = False
VASCULARAGE_REPO = "khalid-saqr/VascularAge"
VASCULARAGE_BRANCH = "phase-01-engine-qualification"
VQ_REPO = "KNOWDYN/VascuQuest"
VQ_SHA = "79891036e61df3096536da8f647f2297b0d88252"
assert ALLOW_BIOLOGICAL_ENDPOINTS is False
print("Phase-1 biological-endpoint guard: LOCKED")


Phase-1 biological-endpoint guard: LOCKED


In [10]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Run this notebook in Google Colab.") from exc
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/VascularAge/phase_01")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["XDG_DATA_HOME"] = str(DRIVE_ROOT / "xdg" / "data")
os.environ["XDG_CACHE_HOME"] = str(DRIVE_ROOT / "xdg" / "cache")
os.environ["XDG_STATE_HOME"] = str(DRIVE_ROOT / "xdg" / "state")
print("Persistent Phase-1 root:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent Phase-1 root: /content/drive/MyDrive/VascularAge/phase_01


In [11]:
VA_ROOT = Path("/content/VascularAge")
VQ_ROOT = Path("/content/VascuQuest")
for path in (VA_ROOT, VQ_ROOT):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(["git","clone","--depth","1","--branch",VASCULARAGE_BRANCH,f"https://github.com/{VASCULARAGE_REPO}.git",str(VA_ROOT)], check=True)
subprocess.run(["git","clone",f"https://github.com/{VQ_REPO}.git",str(VQ_ROOT)], check=True)
subprocess.run(["git","-C",str(VQ_ROOT),"checkout",VQ_SHA], check=True)
assert subprocess.check_output(["git","-C",str(VQ_ROOT),"rev-parse","HEAD"], text=True).strip() == VQ_SHA
contract_bytes = (VA_ROOT/"phase1"/"qualification_contract.json").read_bytes()
contract_sha = hashlib.sha256(contract_bytes).hexdigest()
contract = json.loads(contract_bytes)
assert contract["contract_version"] == "2.0"
assert contract["biological_endpoints_allowed"] is False
print("Observed Phase-1 contract SHA-256:", contract_sha)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(VA_ROOT),"pytest"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(VQ_ROOT)], check=True)
subprocess.run([sys.executable,str(VA_ROOT/"scripts"/"validate_phase1_static.py")], cwd=VA_ROOT, check=True)
print("Repository binding and static validation: PASS")


Observed Phase-1 contract SHA-256: 18cc85b8e192b14acdfff7f08bd684704799e793b34bb558cfede0efbc5f0399
Repository binding and static validation: PASS


In [12]:
cmd = [
    sys.executable, str(VA_ROOT/"scripts"/"phase1_colab.py"),
    "--drive-root", str(DRIVE_ROOT),
    "--repo-root", str(VA_ROOT),
    "--vq-repo-root", str(VQ_ROOT),
]
run = subprocess.run(cmd, cwd=VA_ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr, file=sys.stderr)
assert run.returncode == 0, (
    "Phase 1 qualification failed. Save this notebook as-is; "
    "the exact failure is also persisted in qualification_v2/qualification_failure.json."
)


........                                                                 [100%]

{"report": "/content/drive/MyDrive/VascularAge/phase_01/qualification_v2/vascuquest_tier4_core.json", "status": "passed"}

PHASE 1 QUALIFICATION: PASS
{
  "biological_endpoint_executed": false,
  "contract_sha256": "18cc85b8e192b14acdfff7f08bd684704799e793b34bb558cfede0efbc5f0399",
  "contract_version": "2.0",
  "engine_qualification": "PASS",
  "next_phase_authorized_by_notebook": false,
  "phase": 1,
  "source_qualification": "PASS",
  "status": "PASS",
  "vascularage_repo_commit": "685ef575136ff0628ec2b598e914edce9cdd43e2",
  "vascuquest_commit": "79891036e61df3096536da8f647f2297b0d88252"
}
Qualification bundle: /content/drive/MyDrive/VascularAge/phase_01/qualification_v2



In [13]:
summary_path = DRIVE_ROOT/"qualification_v2"/"qualification_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "PASS"
assert summary["contract_sha256"] == contract_sha
assert summary["biological_endpoint_executed"] is False
print("PHASE 1 EXTERNAL QUALIFICATION CLEARED")
print(json.dumps(summary, indent=2, sort_keys=True))


PHASE 1 EXTERNAL QUALIFICATION CLEARED
{
  "biological_endpoint_executed": false,
  "contract_sha256": "18cc85b8e192b14acdfff7f08bd684704799e793b34bb558cfede0efbc5f0399",
  "contract_version": "2.0",
  "engine_qualification": "PASS",
  "next_phase_authorized_by_notebook": false,
  "phase": 1,
  "source_qualification": "PASS",
  "status": "PASS",
  "vascularage_repo_commit": "685ef575136ff0628ec2b598e914edce9cdd43e2",
  "vascuquest_commit": "79891036e61df3096536da8f647f2297b0d88252"
}
